# Local/HPC Notebook Algorithm Parity Gate

Verifies that paired local and HPC notebooks have byte-identical algorithm cells. Only cells tagged `environment-configuration` may differ. This is a Stage 0 gate and does not execute heavy work.

In [1]:
from pathlib import Path
import hashlib
import json

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "notebooks").exists() and (PROJECT_ROOT / "HPC").exists()) and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "notebooks").exists(), "Could not locate notebook project root"

PAIRS = [
    ("notebooks/pre-processing/predrr_preprocessing.ipynb", "HPC/HPC_notebooks/pre_processing_HPC/predrr_preprocessing.ipynb"),
    ("notebooks/pre-processing/drr_generation.ipynb", "HPC/HPC_notebooks/pre_processing_HPC/drr_generation.ipynb"),
    ("notebooks/data_management/05_vsd_merge_laterality_audit.ipynb", "HPC/HPC_notebooks/pre_processing_HPC/vsd_merge_laterality_audit.ipynb"),
]
ENV_TAG = "environment-configuration"
REPORT_PATH = PROJECT_ROOT / "reports" / "repository_audit" / "notebook_parity_v1.json"

In [2]:
def load_notebook(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def cell_digest(cell):
    payload = {"cell_type": cell["cell_type"], "source": cell.get("source", [])}
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def is_environment_cell(cell):
    return ENV_TAG in cell.get("metadata", {}).get("tags", [])


results = []
for local_rel, hpc_rel in PAIRS:
    local_path, hpc_path = PROJECT_ROOT / local_rel, PROJECT_ROOT / hpc_rel
    entry = {"local": local_rel, "hpc": hpc_rel, "status": "FAIL", "errors": []}
    if not local_path.exists() or not hpc_path.exists():
        entry["errors"].append("missing notebook")
        results.append(entry)
        continue
    local_nb, hpc_nb = load_notebook(local_path), load_notebook(hpc_path)
    if len(local_nb["cells"]) != len(hpc_nb["cells"]):
        entry["errors"].append(f"cell count differs: {len(local_nb['cells'])} vs {len(hpc_nb['cells'])}")
    else:
        for index, (local_cell, hpc_cell) in enumerate(zip(local_nb["cells"], hpc_nb["cells"])):
            tagged = is_environment_cell(local_cell) and is_environment_cell(hpc_cell)
            if is_environment_cell(local_cell) != is_environment_cell(hpc_cell):
                entry["errors"].append(f"cell {index}: environment tag differs")
            elif not tagged and cell_digest(local_cell) != cell_digest(hpc_cell):
                entry["errors"].append(f"cell {index}: algorithm cell differs")
    if not entry["errors"]:
        entry["status"] = "PASS"
    results.append(entry)

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps({"contract": "notebook_parity_v1", "pairs": results}, indent=2), encoding="utf-8")
for row in results:
    print(row["status"], row["local"], row["errors"])
assert all(row["status"] == "PASS" for row in results), "Notebook parity gate failed; inspect report."
print(f"NOTEBOOK PARITY PASS -> {REPORT_PATH}")

PASS notebooks/pre-processing/predrr_preprocessing.ipynb []
PASS notebooks/pre-processing/drr_generation.ipynb []
PASS notebooks/data_management/05_vsd_merge_laterality_audit.ipynb []
NOTEBOOK PARITY PASS -> C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\reports\repository_audit\notebook_parity_v1.json


## Success criterion

All registered preprocessing local/HPC pairs pass. Modeling parity remains in the Claude Code Stage 2 lane. A submission is blocked if an untagged algorithm cell differs, an environment tag exists on only one side, or either notebook is missing.